# 06 - Supplementary figures and tables

**Question.** Do the main duration and segmentation results remain
stable under alternative model and segmentation choices?

| Figure | Analysis |
|---|---|
| S1 | Duration-model comparison and leave-one-fixture-out contrasts |
| S2 | Temporal-sampling and turning-threshold robustness |

Both figures are reconstructed directly from tracked source-data CSVs;
no AWS or private processed cache is required.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# Locate the repository before importing its analysis package. This works when
# Jupyter starts from either the repo root or this notebook directory.
_start = Path.cwd()
ROOT = next(
    path for path in (_start, *_start.parents)
    if (path / "analysis" / "levy_paper").is_dir()
    and (path / "requirements.txt").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.levy_paper.util.publication_notebook_utils import (
    PRIMARY_CACHE_SUFFIX,
    cache_path as make_cache_path,
    csv_shapes,
    display_live_or_frozen,
    file_status,
    hazard_support_summary,
    load_processed_cache,
    order_state_summary,
    panel_inventory,
    publication_paths,
    relative_path,
    resolve_data_mode,
    table_inventory,
    transition_row_sum_audit,
    transport_run_summary,
)

PATHS = publication_paths(ROOT)
LEVY_DIR = PATHS["levy_dir"]
DATA_DIR = PATHS["data_dir"]
PRIMARY_CACHE_DIR = PATHS["primary_cache_dir"]
FINAL_FIGURES = PATHS["final_figures"]
SOURCE_DATA = PATHS["source_data"]
SUPPLEMENT = PATHS["supplement"]
CACHE_SUFFIX = PRIMARY_CACHE_SUFFIX

# DATA_MODE options:
#   "auto"     use processed caches when all required files exist;
#              otherwise use tracked reviewer tables/frozen figures
#   "cache"    require processed caches and fail clearly if they are missing
#   "reviewer" use only tracked public artefacts
DATA_MODE = "auto"
BUILD_FIGURE = True
SAVE_FIGURE_OUTPUTS = True
DISPLAY_FROZEN_OUTPUT = True
REBUILD_CACHE_FROM_AWS = False
REFIT_FIGURE4_FIGURE5_MODELS = False
USE_VERSIONED_FINAL_FIGURE4_FIT = True


def rel(path):
    return relative_path(path, ROOT)


def show_file_status(paths):
    return file_status(paths, ROOT)


def show_csv_shapes(paths):
    return csv_shapes(paths, ROOT)


def cache_path(stem):
    return make_cache_path(PRIMARY_CACHE_DIR, stem, CACHE_SUFFIX)


def show_figure(fig, frozen_path, width=1100):
    return display_live_or_frozen(
        fig,
        frozen_path,
        display_frozen=DISPLAY_FROZEN_OUTPUT,
        width=width,
    )

## 1. Source-data and formal-table inventory

In [ ]:
source_files = sorted((SUPPLEMENT / "source_data").glob("*.csv"))
table_files = sorted((SUPPLEMENT / "tables").glob("*.csv"))
display(show_csv_shapes(source_files))
display(show_csv_shapes(table_files))

## 2. Figure S1 inputs - duration-model comparison

In [ ]:
s1_curves = pd.read_csv(SUPPLEMENT / "source_data" / "figureS1_panelA_model_curves.csv")
s1_contrasts = pd.read_csv(SUPPLEMENT / "source_data" / "figureS1_panelB_lofo_contrasts.csv")
duration_gof = pd.read_csv(SUPPLEMENT / "source_data" / "duration_model_gof_final.csv")
display(panel_inventory({
    "S1 model curves": s1_curves,
    "S1 LOFO contrasts": s1_contrasts,
    "duration goodness of fit": duration_gof,
}))
display(duration_gof)
display(s1_contrasts.head(12))

## 3. Construct Figure S1

In [ ]:
from analysis.levy_paper.scripts import create_supplementary_figures_from_source as supplement_figures
from analysis.levy_paper.util.paper_utils import configure_paper_plotting

configure_paper_plotting(base=10)
fig_s1 = None
if BUILD_FIGURE:
    fig_s1, paths_s1 = supplement_figures.figure_s1(close_figure=False)
display_mode_s1 = show_figure(fig_s1, SUPPLEMENT / "figures" / "figureS1_duration_model_comparison.png")
print("figureS1_display_mode", display_mode_s1)

## 4. Figure S2 inputs - segmentation robustness

In [ ]:
s2_source = pd.read_csv(SUPPLEMENT / "source_data" / "figureS2_segmentation_robustness_source.csv")
s2_table = pd.read_csv(SUPPLEMENT / "source_data" / "tableS_segmentation_robustness_final.csv")
display(panel_inventory({"S2 plotted source": s2_source, "S2 numerical table": s2_table}))
display(s2_table)

## 5. Construct Figure S2

In [ ]:
fig_s2 = None
if BUILD_FIGURE:
    fig_s2, paths_s2 = supplement_figures.figure_s2(close_figure=False)
display_mode_s2 = show_figure(fig_s2, SUPPLEMENT / "figures" / "figureS2_segmentation_robustness_final.png")
print("figureS2_display_mode", display_mode_s2)